In [1]:
from pathlib import Path
import re

import pandas as pd
import plotly.express as px

BASE_DIR  = Path.cwd()
DATA_FILE = BASE_DIR / "number_of_affected_people.xlsx"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Could not find the Excel file at {DATA_FILE}.\n"
        "Place it in the same folder as this notebook."
    )

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE, OUTPUT_DIR

(WindowsPath('d:/Disaster/number_of_affected_people.xlsx'),
 WindowsPath('d:/Disaster/outputs'))

## Inspecting the Raw Workbook

The PSA workbook uses a reporting layout, not a ready-made CSV table.

- Rows 0–2 contain the table title and date range.
- Row 4 contains the **years** (merged cells: 2015 to 2023).
- Row 5 contains the **sub-metrics** (Families, Persons) repeated per year.
- Column A contains the **hazard type** label.
- Lower rows contain footnotes and the source attribution.

In [2]:
raw_df = pd.read_excel(DATA_FILE, header=None)
print(f"Raw shape: {raw_df.shape}")
raw_df.head(10)

Raw shape: (77, 19)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,Table 4.9.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NUMBER OF AFFECTED PEOPLE DUE TO NATURAL EXTRE...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015 to 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Hazard Type,2015,NaN,2016,NaN,2017,NaN,2018,NaN,2019,NaN,2020,NaN,2021,NaN,2022,NaN,2023,NaN
5,NaN,Families,Persons,Families,Persons,Families,Persons,Families,Persons,Families,Persons,Families,Persons,Families,Persons,Families,Persons,Families,Persons
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Total,1174159,5096031,1570891,7037858,1088770,4859997,2528709,10449293,2495808,10835560,2336359,9527305,3854289,14009977,3039636,11700593,3068390,12063875
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,African Swine Fever,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Extracting the Usable Table



In [3]:
YEAR_ROW    = 4   # row index containing year labels
SUBTYPE_ROW = 5   # row index containing Families / Persons
DATA_START  = 7   # first row of actual hazard data

year_row    = raw_df.iloc[YEAR_ROW][1:].ffill()    # forward-fill merged year cells
subtype_row = raw_df.iloc[SUBTYPE_ROW][1:]

# Build column names: "2015_Families", "2015_Persons", ...
columns = ["hazard_type"]
for yr, sub in zip(year_row, subtype_row):
    if pd.notna(yr) and pd.notna(sub):
        columns.append(f"{int(yr)}_{sub}")

print("Columns built:")
print(columns)

Columns built:
['hazard_type', '2015_Families', '2015_Persons', '2016_Families', '2016_Persons', '2017_Families', '2017_Persons', '2018_Families', '2018_Persons', '2019_Families', '2019_Persons', '2020_Families', '2020_Persons', '2021_Families', '2021_Persons', '2022_Families', '2022_Persons', '2023_Families', '2023_Persons']


In [4]:
numeric_cols = [c for c in columns if c != "hazard_type"]
first_num    = numeric_cols[0]

wide_df = raw_df.iloc[DATA_START:, :len(columns)].copy()
wide_df.columns = columns

# Keep rows that have a hazard label and at least one numeric value
wide_df = wide_df[
    wide_df["hazard_type"].notna() &
    wide_df[first_num].notna()
].copy()
wide_df.reset_index(drop=True, inplace=True)

print(f"Extracted shape: {wide_df.shape}")
wide_df.head()

Extracted shape: (52, 19)


,hazard_type,2015_Families,2015_Persons,2016_Families,2016_Persons,2017_Families,2017_Persons,2018_Families,2018_Persons,2019_Families,2019_Persons,2020_Families,2020_Persons,2021_Families,2021_Persons,2022_Families,2022_Persons,2023_Families,2023_Persons
0,Total,1174159,5096031,1570891,7037858,1088770,4859997,2528709,10449293,2495808,10835560,2336359,9527305,3854289,14009977,3039636,11700593,3068390,12063875
1,African Swine Fever,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Bird Strikes,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Disease/Epidemic Outbreak,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fish Kill,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Cleaning and Reshaping the Data

We clean the hazard type labels, remove the Total summary row, convert all numeric columns, then melt the wide table into a tidy long format.

In [5]:
# Clean hazard_type labels
wide_df["hazard_type"] = (
    wide_df["hazard_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+[A-I2]\s*$", "", regex=True)   # remove trailing footnote letters
    .str.replace(r"\n", " ", regex=True)               # flatten line breaks inside labels
    .str.strip()
)

# Drop the Total summary row and keep hazard-level rows only
wide_df = wide_df[wide_df["hazard_type"] != "Total"].copy()
wide_df.reset_index(drop=True, inplace=True)

# Convert numeric columns
for col in numeric_cols:
    wide_df[col] = pd.to_numeric(wide_df[col], errors="coerce").fillna(0).astype(int)

print(f"Clean wide shape: {wide_df.shape}")
wide_df.head()

Clean wide shape: (51, 19)


,hazard_type,2015_Families,2015_Persons,2016_Families,2016_Persons,2017_Families,2017_Persons,2018_Families,2018_Persons,2019_Families,2019_Persons,2020_Families,2020_Persons,2021_Families,2021_Persons,2022_Families,2022_Persons,2023_Families,2023_Persons
0,African Swine Fever,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Bird Strikes,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Disease/Epidemic Outbreak,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fish Kill,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Pest Infestation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Melt to long format
long_df = wide_df.melt(
    id_vars="hazard_type",
    var_name="series_key",
    value_name="count"
)

# Split "2015_Families" → year=2015, metric=Families
series_parts = long_df["series_key"].str.extract(r"^(?P<year>\d{4})_(?P<metric>.+)$")
long_df = pd.concat([long_df, series_parts], axis=1).drop(columns="series_key")
long_df["year"]  = long_df["year"].astype(int)
long_df["count"] = pd.to_numeric(long_df["count"], errors="coerce").fillna(0).astype(int)

# Pivot metric back into two clean columns
tidy_df = long_df.pivot_table(
    index=["hazard_type", "year"],
    columns="metric",
    values="count",
    aggfunc="first"
).reset_index()
tidy_df.columns.name = None
tidy_df = tidy_df.rename(columns={"Families": "families_affected", "Persons": "persons_affected"})
tidy_df = tidy_df.sort_values(["hazard_type", "year"]).reset_index(drop=True)

# Derived columns
tidy_df["avg_persons_per_family"] = (
    tidy_df["persons_affected"].astype(float)
    / tidy_df["families_affected"].replace(0, float("nan"))
).round(2)

tidy_df["persons_yoy_change"] = (
    tidy_df.groupby("hazard_type")["persons_affected"]
    .diff()
    .fillna(0)
    .astype(int)
)

tidy_df["had_impact"] = tidy_df["persons_affected"] > 0

print(f"Tidy shape: {tidy_df.shape}")
tidy_df.head(10)

Tidy shape: (459, 7)


,hazard_type,year,families_affected,persons_affected,avg_persons_per_family,persons_yoy_change,had_impact
0,African Swine Fever,2015,0,0,NaN,0,False
1,African Swine Fever,2016,0,0,NaN,0,False
2,African Swine Fever,2017,0,0,NaN,0,False
3,African Swine Fever,2018,0,0,NaN,0,False
4,African Swine Fever,2019,0,0,NaN,0,False
5,African Swine Fever,2020,0,0,NaN,0,False
6,African Swine Fever,2021,0,0,NaN,0,False
7,African Swine Fever,2022,0,0,NaN,0,False
8,African Swine Fever,2023,0,0,NaN,0,False
9,Bird Strikes,2015,0,0,NaN,0,False


## Exploring the Cleaned Dataset

In [7]:
overview = {
    "row_count"         : len(tidy_df),
    "year_min"          : int(tidy_df["year"].min()),
    "year_max"          : int(tidy_df["year"].max()),
    "unique_hazard_types": int(tidy_df["hazard_type"].nunique()),
    "total_persons_all_years": int(tidy_df["persons_affected"].sum()),
}
pd.Series(overview)

row_count                       459
year_min                       2015
year_max                       2023
unique_hazard_types              51
total_persons_all_years    85580489
dtype: int64

In [8]:
# Total persons affected per year across all hazard types
yearly_totals = (
    tidy_df.groupby("year")[["families_affected", "persons_affected"]]
    .sum()
    .reset_index()
)
yearly_totals

,year,families_affected,persons_affected
0,2015,1174159,5096031
1,2016,1570891,7037858
2,2017,1088770,4859997
3,2018,2528709,10449293
4,2019,2495808,10835560
5,2020,2336359,9527305
6,2021,3854289,14009977
7,2022,3039636,11700593
8,2023,3068390,12063875


In [9]:
# Top 10 hazard types by total persons affected across all years
summary = (
    tidy_df.groupby("hazard_type")["persons_affected"]
    .agg(
        total_persons="sum",
        max_year_persons="max",
        years_with_impact=lambda x: (x > 0).sum()
    )
    .sort_values("total_persons", ascending=False)
    .reset_index()
)
summary.head(10)

,hazard_type,total_persons,max_year_persons,years_with_impact
0,Tropical Cyclones,70834024,12765120,9
1,Southwest Monsoon (SWM),2717481,1285150,3
2,Earthquake,2580522,957869,6
3,Drought/ El Niño Phenomenon/ Dry Spell,2432804,2432804,1
4,LPA and Shearline,2342110,1157487,3
5,"LPA, Northeast Monsoon, and Shearline",2143347,2143347,1
6,Shearline,1019841,743956,3
7,Volcanic Activity/ Volcanic Smog,580204,417230,5
8,Rain,454934,454934,1
9,Low Pressure Area (LPA),273577,273577,1


## Building and Saving Reusable Charts

Charts are saved as HTML files for reliability, matching the reference notebook approach.

In [10]:
def save_chart_html(fig, filename: str) -> Path:
    output_path = OUTPUT_DIR / filename
    fig.write_html(output_path, include_plotlyjs="cdn")
    return output_path

In [11]:
# Chart 1: Total persons affected per year
line_fig = px.line(
    yearly_totals,
    x="year",
    y="persons_affected",
    markers=True,
    title="Total Persons Affected by Natural Hazards per Year (2015–2023)",
    labels={
        "persons_affected": "Persons Affected",
        "year": "Year",
    },
)
line_fig.update_layout(template="plotly_white")
line_path = save_chart_html(line_fig, "total_persons_per_year.html")
line_fig

In [12]:
# Chart 2: Top 10 hazard types by total persons affected
top10 = summary.head(10)
bar_fig = px.bar(
    top10,
    x="total_persons",
    y="hazard_type",
    orientation="h",
    title="Top 10 Hazard Types by Total Persons Affected (2015–2023)",
    labels={
        "total_persons": "Total Persons Affected",
        "hazard_type": "Hazard Type",
    },
)
bar_fig.update_layout(template="plotly_white", yaxis=dict(autorange="reversed"))
bar_path = save_chart_html(bar_fig, "top10_hazards.html")
bar_fig

In [13]:
# Chart 3: Trend for top 5 hazard types over time
top5_hazards = summary.head(5)["hazard_type"].tolist()
top5_df = tidy_df[tidy_df["hazard_type"].isin(top5_hazards)].copy()

trend_fig = px.line(
    top5_df,
    x="year",
    y="persons_affected",
    color="hazard_type",
    markers=True,
    title="Persons Affected by Top 5 Hazard Types Over Time (2015–2023)",
    labels={
        "persons_affected": "Persons Affected",
        "year": "Year",
        "hazard_type": "Hazard Type",
    },
)
trend_fig.update_layout(template="plotly_white")
trend_path = save_chart_html(trend_fig, "top5_trend.html")
trend_fig

## Exporting Cleaned Files

We export both the wide cleaned table and the final tidy dataset so they can be reused in the Streamlit dashboard.

In [14]:
wide_out    = OUTPUT_DIR / "cleaned_wide.csv"
tidy_out    = OUTPUT_DIR / "cleaned_tidy.csv"
summary_out = OUTPUT_DIR / "summary_by_hazard.csv"

wide_df.to_csv(wide_out,    index=False)
tidy_df.to_csv(tidy_out,    index=False)
summary.to_csv(summary_out, index=False)

pd.Series({
    "cleaned_wide_csv"    : str(wide_out),
    "cleaned_tidy_csv"    : str(tidy_out),
    "summary_csv"         : str(summary_out),
    "line_chart_html"     : str(line_path),
    "bar_chart_html"      : str(bar_path),
    "trend_chart_html"    : str(trend_path),
})

cleaned_wide_csv               d:\Disaster\outputs\cleaned_wide.csv
cleaned_tidy_csv               d:\Disaster\outputs\cleaned_tidy.csv
summary_csv               d:\Disaster\outputs\summary_by_hazard.csv
line_chart_html     d:\Disaster\outputs\total_persons_per_year.html
bar_chart_html               d:\Disaster\outputs\top10_hazards.html
trend_chart_html                d:\Disaster\outputs\top5_trend.html
dtype: str